In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path.home() / "Downloads" / "transformer_data"

train_df = pd.read_csv(DATA_DIR / "Train.csv")
val_df = pd.read_csv(DATA_DIR / "Validation.csv")
test_df = pd.read_csv(DATA_DIR / "Test.csv")

existing_train = pd.read_csv("train.csv")
existing_val = pd.read_csv("val.csv")
existing_test = pd.read_csv("test.csv")

print(f"Original Train: {train_df.shape}")
print(f"Original Validation: {val_df.shape}")
print(f"Original Test: {test_df.shape}")
print(f"\nExisting train: {existing_train.shape}")
print(f"Existing val:   {existing_val.shape}")
print(f"Existing test:  {existing_test.shape}")

Original Train: (50420, 6)
Original Validation: (7047, 6)
Original Test: (6981, 6)

Existing train: (12190, 6)
Existing val:   (1524, 6)
Existing test:  (1524, 6)


In [2]:
TARGET_LABELS = [
    "Turning Hand Clockwise",
    "Turning Hand Counterclockwise",
    "Zooming In With Two Fingers",
    "Zooming Out With Two Fingers",
]

train = train_df.copy()
val = val_df.copy()
test = test_df.rename(columns={"id": "video_id"}).copy()

combined_df = pd.concat([train, val, test], ignore_index=True)
filtered_df = combined_df[combined_df["label"].isin(TARGET_LABELS)].copy()

OUTPUT_PATH = Path("additional_data.csv")
filtered_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(filtered_df)} rows to {OUTPUT_PATH.resolve()}")
print(filtered_df.groupby(["label", "label_id"]).size())
print("\nLabel counts:")
print(filtered_df["label"].value_counts())

Saved 7261 rows to /Users/aryamanwade/Desktop/ml_projects/remote_desktop/remote-desktop/additional_data.csv
label                          label_id
Turning Hand Clockwise         21.0        1523
Turning Hand Counterclockwise  22.0        1572
Zooming In With Two Fingers    24.0        2058
Zooming Out With Two Fingers   26.0        2108
dtype: int64

Label counts:
label
Zooming Out With Two Fingers     2108
Zooming In With Two Fingers      2058
Turning Hand Counterclockwise    1572
Turning Hand Clockwise           1523
Name: count, dtype: int64


In [3]:
train_split, temp_split = train_test_split(
    filtered_df,
    test_size=0.2,
    stratify=filtered_df["label"],
    random_state=42,
)

val_split, test_split = train_test_split(
    temp_split,
    test_size=0.5,
    stratify=temp_split["label"],
    random_state=42,
)

print(f"New train: {len(train_split)} ({len(train_split) / len(filtered_df):.1%})")
print(f"New val:   {len(val_split)} ({len(val_split) / len(filtered_df):.1%})")
print(f"New test:  {len(test_split)} ({len(test_split) / len(filtered_df):.1%})")

print("\nNew train label distribution:")
print(train_split["label"].value_counts(normalize=True).round(3))
print("\nNew val label distribution:")
print(val_split["label"].value_counts(normalize=True).round(3))
print("\nNew test label distribution:")
print(test_split["label"].value_counts(normalize=True).round(3))

New train: 5808 (80.0%)
New val:   726 (10.0%)
New test:  727 (10.0%)

New train label distribution:
label
Zooming Out With Two Fingers     0.290
Zooming In With Two Fingers      0.283
Turning Hand Counterclockwise    0.217
Turning Hand Clockwise           0.210
Name: proportion, dtype: float64

New val label distribution:
label
Zooming Out With Two Fingers     0.291
Zooming In With Two Fingers      0.284
Turning Hand Counterclockwise    0.216
Turning Hand Clockwise           0.209
Name: proportion, dtype: float64

New test label distribution:
label
Zooming Out With Two Fingers     0.290
Zooming In With Two Fingers      0.283
Turning Hand Counterclockwise    0.216
Turning Hand Clockwise           0.210
Name: proportion, dtype: float64


In [4]:
updated_train = pd.concat([existing_train, train_split], ignore_index=True)
updated_val = pd.concat([existing_val, val_split], ignore_index=True)
updated_test = pd.concat([existing_test, test_split], ignore_index=True)

updated_train.to_csv("train.csv", index=False)
updated_val.to_csv("val.csv", index=False)
updated_test.to_csv("test.csv", index=False)

print(f"Updated train: {len(updated_train)} (+{len(train_split)})")
print(f"Updated val:   {len(updated_val)} (+{len(val_split)})")
print(f"Updated test:  {len(updated_test)} (+{len(test_split)})")

print("\nUpdated train label distribution:")
print(updated_train["label"].value_counts())
print("\nUpdated val label distribution:")
print(updated_val["label"].value_counts())
print("\nUpdated test label distribution:")
print(updated_test["label"].value_counts())

Updated train: 17998 (+5808)
Updated val:   2250 (+726)
Updated test:  2251 (+727)

Updated train label distribution:
label
No gesture                       5749
Zooming Out With Two Fingers     1686
Swiping Down                     1658
Zooming In With Two Fingers      1646
Swiping Left                     1607
Swiping Up                       1607
Swiping Right                    1569
Turning Hand Counterclockwise    1258
Turning Hand Clockwise           1218
Name: count, dtype: int64

Updated val label distribution:
label
No gesture                       719
Zooming Out With Two Fingers     211
Swiping Down                     207
Zooming In With Two Fingers      206
Swiping Left                     201
Swiping Up                       201
Swiping Right                    196
Turning Hand Counterclockwise    157
Turning Hand Clockwise           152
Name: count, dtype: int64

Updated test label distribution:
label
No gesture                       719
Zooming Out With Two Fingers     

## MediaPipe landmark extraction (additional data)

Extract hand landmarks from raw frames for videos in `additional_data.csv`. Raw `.npy` sequences are saved under `additional_data/` (before interpolation).

In [5]:
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

DATA_DIR = Path.home() / "Downloads" / "transformer_data"
SOURCE_SPLITS = ("Train", "Validation", "Test")
NUM_FRAMES = 37

MODEL_PATH = Path("mediapipe_model//hand_landmarker.task")
ADDITIONAL_DATA_DIR = Path("additional_data")
SPLIT_FOLDERS = {
    "train": ADDITIONAL_DATA_DIR / "train",
    "val": ADDITIONAL_DATA_DIR / "validate",
    "test": ADDITIONAL_DATA_DIR / "test",
}

additional_ids = set(pd.read_csv("additional_data.csv")["video_id"])
_split_lookup = {}
for split_name, csv_path in [
    ("train", "train.csv"),
    ("val", "val.csv"),
    ("test", "test.csv"),
]:
    for video_id in pd.read_csv(csv_path)["video_id"]:
        if video_id in additional_ids:
            _split_lookup[video_id] = split_name

_landmarker = None

In [6]:
def find_video(video_id):
    """Return the folder path for a video id under Train, Validation, or Test."""
    for split in SOURCE_SPLITS:
        video_path = DATA_DIR / split / str(video_id)
        if video_path.is_dir():
            return video_path
    raise FileNotFoundError(
        f"Video {video_id} not found in {', '.join(SOURCE_SPLITS)}"
    )


def load_frames(video_path, num_frames=NUM_FRAMES):
    """Load the JPG frames for a video folder."""
    frames = []
    for i in range(1, num_frames + 1):
        frame_path = video_path / f"{i:05d}.jpg"
        frame = cv2.imread(str(frame_path))
        if frame is None:
            raise FileNotFoundError(f"Missing frame: {frame_path}")
        frames.append(frame)
    return frames


def _get_landmarker():
    global _landmarker
    if _landmarker is None:
        MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not MODEL_PATH.exists():
            url = (
                "https://storage.googleapis.com/mediapipe-models/"
                "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
            )
            urllib.request.urlretrieve(url, MODEL_PATH)

        options = vision.HandLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path=str(MODEL_PATH)),
            running_mode=vision.RunningMode.IMAGE,
            num_hands=1,
            min_hand_detection_confidence=0.3,
        )
        _landmarker = vision.HandLandmarker.create_from_options(options)
    return _landmarker


def extract_landmarks(frames):
    """Run MediaPipe hand landmarks on each frame. Returns shape (37, 63)."""
    landmarker = _get_landmarker()
    landmarks = []

    for frame in frames:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)

        frame_landmarks = np.zeros((21, 3), dtype=np.float32)
        if result.hand_landmarks:
            for i, lm in enumerate(result.hand_landmarks[0]):
                frame_landmarks[i] = [lm.x, lm.y, lm.z]
        landmarks.append(frame_landmarks)

    landmarks = np.stack(landmarks)
    return landmarks.reshape(NUM_FRAMES, 63)


def save_sequence(video_id, landmarks):
    """Save landmarks to additional_data/train, validate, or test."""
    split = _split_lookup.get(video_id)
    if split is None:
        raise ValueError(
            f"video_id {video_id} not found in train.csv, val.csv, or test.csv"
        )

    out_dir = SPLIT_FOLDERS[split]
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f"{video_id}.npy", landmarks)

In [ ]:
from tqdm import tqdm

for split_folder in SPLIT_FOLDERS.values():
    split_folder.mkdir(parents=True, exist_ok=True)

video_ids = sorted(additional_ids)
failed = []

for video_id in tqdm(video_ids, desc="Processing additional videos"):
    split = _split_lookup[video_id]
    out_path = SPLIT_FOLDERS[split] / f"{video_id}.npy"
    if out_path.exists():
        continue

    try:
        video_path = find_video(video_id)
        frames = load_frames(video_path)
        landmarks = extract_landmarks(frames)
        save_sequence(video_id, landmarks)
    except Exception as e:
        failed.append((video_id, str(e)))

print(f"Done. Saved {len(video_ids) - len(failed)} sequences to {ADDITIONAL_DATA_DIR.resolve()}.")
if failed:
    print(f"Failed: {len(failed)}")
    for video_id, error in failed[:10]:
        print(f"  {video_id}: {error}")

Processing additional videos:   0%|          | 0/7261 [00:00<?, ?it/s]I0000 00:00:1782842232.410793 32653733 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
I0000 00:00:1782842232.520389 32653733 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1782842232.528668 32653737 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782842232.539254 32653741 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782842232.551065 32653736 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
Processing additional videos:   5%|▍         | 3

Done. Saved 7261 sequences to /Users/aryamanwade/Desktop/ml_projects/remote_desktop/remote-desktop/additional_data.


E0000 00:00:1782844633.287383 32653734 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-30T14:43:13.247186-04:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782844693.293483 32653734 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-30T14:43:13.247186-04:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782844753.299568 32653734 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-30T14:43:13.247186-04:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782844813.301332 32653734 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not v

## Interpolation

Fill missing landmark gaps in `additional_data/`, then append interpolated sequences to the matching splits in `final_data/`.

In [8]:
FINAL_DATA_DIR = Path("final_data")

INTERP_SPLITS = {
    "train": "train",
    "val": "validate",
    "test": "test",
}


def hand_detected_mask(seq):
    """True for frames with at least one non-zero landmark."""
    return np.any(seq != 0, axis=1)


def gesture_window(mask):
    """Return inclusive (start, end) of first/last hand detection, or (None, None)."""
    if not mask.any():
        return None, None
    start = int(np.argmax(mask))
    end = int(len(mask) - 1 - np.argmax(mask[::-1]))
    return start, end


def interpolate_gaps(seq):
    """Linearly fill zero-frame gaps inside the gesture window."""
    seq = seq.copy()
    detected = hand_detected_mask(seq)
    start_win, end_win = gesture_window(detected)
    if start_win is None:
        return seq

    i = start_win
    while i <= end_win:
        if detected[i]:
            i += 1
            continue

        gap_start = i
        while i <= end_win and not detected[i]:
            i += 1
        gap_end = i
        gap = gap_end - gap_start

        if gap <= 0 or gap_end > end_win:
            continue

        prev_frame = seq[gap_start - 1]
        next_frame = seq[gap_end]
        start = gap_start - 1

        for k in range(1, gap + 1):
            alpha = k / (gap + 1)
            seq[start + k] = (1 - alpha) * prev_frame + alpha * next_frame

    return seq

In [9]:
from tqdm import tqdm

appended = 0
skipped = 0

for split_name, folder in INTERP_SPLITS.items():
    in_dir = ADDITIONAL_DATA_DIR / folder
    out_dir = FINAL_DATA_DIR / folder
    out_dir.mkdir(parents=True, exist_ok=True)

    npy_files = sorted(in_dir.glob("*.npy"))
    for npy_path in tqdm(npy_files, desc=f"Interpolating {split_name}"):
        out_path = out_dir / npy_path.name
        if out_path.exists():
            skipped += 1
            continue

        seq = np.load(npy_path)
        interpolated = interpolate_gaps(seq)
        np.save(out_path, interpolated.astype(np.float32))
        appended += 1

print(f"Interpolation complete. Appended {appended} sequences to {FINAL_DATA_DIR.resolve()}.")
if skipped:
    print(f"Skipped {skipped} sequences already present in final_data.")

Interpolating test: 100%|██████████| 727/727 [00:00<00:00, 3408.31it/s]

Interpolation complete. Appended 7261 sequences to /Users/aryamanwade/Desktop/ml_projects/remote_desktop/remote-desktop/final_data.
